In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths

ROOT = Path("..").resolve()

OUTPUTS_DIR = ROOT / "outputs"


OLD_RUN = "local_1260_260629"
NEW_RUN = "local_1260_260722"

OLD_PATH = Path(
    f"{OUTPUTS_DIR}/phase5_{OLD_RUN}/metrics/phase5_{OLD_RUN}_xai_concept_metrics.csv"
)

NEW_PATH = Path(
    f"{OUTPUTS_DIR}/phase5_{NEW_RUN}/metrics/phase5_{NEW_RUN}_xai_concept_metrics.csv"
)

OUTPUT_DIR = Path(f"{OUTPUTS_DIR}/results/comparisons")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load image-level results


old_df = pd.read_csv(OLD_PATH)
new_df = pd.read_csv(NEW_PATH)

print("Old shape:", old_df.shape)
print("New shape:", new_df.shape)
print("Old columns:", old_df.columns.tolist())

In [ ]:
KEY_COLUMNS = ["stem", "xai_method", "concept"]
METRIC_COLUMNS = ["iou", "dice", "sir", "tixai"]

# Keep only the required columns
old_compare = old_df[KEY_COLUMNS + METRIC_COLUMNS].copy()
new_compare = new_df[KEY_COLUMNS + METRIC_COLUMNS].copy()

merged = old_compare.merge(
    new_compare,
    on=KEY_COLUMNS,
    how="inner",
    suffixes=("_old", "_new"),
    validate="one_to_one",
)

print("Matched rows:", len(merged))
print("Old unmatched rows:", len(old_compare) - len(merged))
print("New unmatched rows:", len(new_compare) - len(merged))

In [ ]:
for metric in METRIC_COLUMNS:
    merged[f"delta_{metric}"] = (
        merged[f"{metric}_new"] - merged[f"{metric}_old"]
    )

    merged[f"abs_delta_{metric}"] = (
        merged[f"delta_{metric}"].abs()
    )



    summary_rows = []

for (method, concept), group in merged.groupby(
    ["xai_method", "concept"]
):
    for metric in METRIC_COLUMNS:
        old_values = group[f"{metric}_old"]
        new_values = group[f"{metric}_new"]
        differences = group[f"delta_{metric}"]
        absolute_differences = group[f"abs_delta_{metric}"]

        summary_rows.append(
            {
                "xai_method": method,
                "concept": concept,
                "metric": metric,
                "n": len(group),
                "old_mean": old_values.mean(),
                "new_mean": new_values.mean(),
                "mean_difference": differences.mean(),
                "median_difference": differences.median(),
                "mean_absolute_difference": (
                    absolute_differences.mean()
                ),
                "difference_sd": differences.std(),
                "percent_increased": (
                    differences.gt(0).mean() * 100
                ),
                "spearman_correlation": old_values.corr(
                    new_values,
                    method="spearman",
                ),
            }
        )

paired_summary = pd.DataFrame(summary_rows)

paired_summary = paired_summary.sort_values(
    ["concept", "xai_method", "metric"]
)

display(paired_summary)

summary_path = (
    OUTPUT_DIR
    / "phase5_old_vs_grouped_paired_summary.csv"
)

paired_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)

In [ ]:
KEEP_CONCEPTS = [
    "asymmetry",
    "colour_heterogeneity",
    "border_w16_dil6_sigma10_dist",
]



for metric in METRIC_COLUMNS:

    merged_filtered = merged[
        merged["concept"].isin(KEEP_CONCEPTS)
    ].copy()


    print(sorted(merged["concept"].dropna().unique()))

    plot_df = merged_filtered.copy()

    plot_df["method_concept"] = (
        plot_df["xai_method"].astype(str)
        + "\n"
        + plot_df["concept"].astype(str)
    )

    groups = [
        group[f"delta_{metric}"].dropna().values
        for _, group in plot_df.groupby(
            "method_concept",
            sort=False,
        )
    ]

    labels = [
        name
        for name, _ in plot_df.groupby(
            "method_concept",
            sort=False,
        )
    ]

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.boxplot(
        groups,
        labels=labels,
        showfliers=False,
    )

    ax.axhline(0, linewidth=1)

    ax.set_ylabel(f"Change in {metric}: grouped − original")
    ax.set_xlabel("XAI method and pseudo-concept")
    ax.set_title(
        "Paired change in XAI alignment after lesion-grouped training"
    )

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    figure_path = (
        OUTPUT_DIR
        / f"phase5_paired_{metric}_difference_boxplot.png"
    )

    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print("Saved:", figure_path)


In [ ]:
print(
    merged.groupby(["xai_method", "concept"])["delta_dice"]
    .agg(["count", "mean", "median", "min", "max"])
)

print(OLD_PATH)
print(NEW_PATH)

print(
    old_df[old_df["xai_method"] == "gradcam"]["dice"].head()
)

print(
    new_df[new_df["xai_method"] == "gradcam"]["dice"].head()
)